# Stage 3: Wall-to-Wall Inference and Carbon Stock Derivation

**Author** : Muhammad Wahyu Ramadhan
**GitHub**  : github.com/mwahyur46
**LinkedIn**: linkedin.com/in/mwahyur

Applies the trained Stage 1 (CH) and Stage 2 (AGB) models to the
wall-to-wall predictor stack exported from GEE, then derives carbon stock
via deterministic raster math. Outputs three final GeoTIFF maps masked to
the mangrove extent.

**Workflow steps:**
1. Load the base predictor stack (`base_stack_west_kalimantan.tif`)
2. Load the trained CH and AGB models
3. Predict wall-to-wall canopy height (Stage 1)
4. Append `CH_m` to the predictor stack, predict wall-to-wall AGB (Stage 2)
5. Derive carbon stock: AGB x 0.451 (Stage 3, IPCC 2014 Tier 1 factor)
6. Mask all outputs to the GMW mangrove extent
7. Compute distribution statistics and total carbon stock
8. Export final rasters to `outputs/`

**Carbon stock reference:**
IPCC (2014). 2013 Supplement to the 2006 IPCC Guidelines for National
Greenhouse Gas Inventories: Wetlands. Chapter 4 (Coastal Wetlands).
Tier 1 default carbon fraction for mangrove AGB: 0.451 (45.1%).
https://www.ipcc-nggip.iges.or.jp/public/wetlands/


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/src')

import joblib
import numpy as np
import rasterio
import matplotlib.pyplot as plt

import raster_utils as ru


## 1. Load Predictor Stack


In [ ]:
STACK_PATH = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/data/raw/base_stack_west_kalimantan.tif'

stack_array, profile, band_names = ru.load_stack(STACK_PATH)


## 2. Load Trained Models


In [ ]:
CH_MODEL_PATH  = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/models/ch_rf_model.pkl'
AGB_MODEL_PATH = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/models/agb_rf_model.pkl'

ch_model  = joblib.load(CH_MODEL_PATH)
agb_model = joblib.load(AGB_MODEL_PATH)

print(f'Loaded CH model : {CH_MODEL_PATH}')
print(f'Loaded AGB model: {AGB_MODEL_PATH}')


## 3. Stage 1 -- Wall-to-Wall Canopy Height

Flatten the predictor stack to valid pixels, predict, and reconstruct the
2D canopy height raster.


In [ ]:
X_flat, valid_mask, original_shape = ru.stack_to_dataframe(stack_array, band_names)

ch_pred_flat = ch_model.predict(X_flat)

ch_map = np.full(original_shape, np.nan, dtype='float32')
ch_map[valid_mask] = ch_pred_flat

print('Canopy height map generated.')
print(f'Shape: {ch_map.shape}')


## 4. Stage 2 -- Wall-to-Wall AGB

Append the predicted CH map as an additional band, then predict AGB.


In [ ]:
# Append CH_m as an extra band to the valid-pixel feature table
ch_flat_valid = ch_map[valid_mask].reshape(-1, 1)
X_flat_agb = np.hstack([X_flat, ch_flat_valid])

agb_pred_flat = agb_model.predict(X_flat_agb)

agb_map = np.full(original_shape, np.nan, dtype='float32')
agb_map[valid_mask] = agb_pred_flat

print('AGB map generated.')
print(f'Shape: {agb_map.shape}')


## 5. Stage 3 -- Carbon Stock Derivation

Deterministic raster math: Carbon (Mg C/ha) = AGB (Mg/ha) x 0.451
(IPCC 2014 Wetlands Supplement, Chapter 4, Tier 1 mangrove default).


In [ ]:
CARBON_FRACTION = 0.451  # IPCC (2014), Chapter 4

carbon_map = ru.compute_carbon_stock(agb_map, carbon_fraction=CARBON_FRACTION)

print('Carbon stock map derived.')


## 6. Mangrove Mask Application

Apply the GMW mangrove mask (exported alongside the predictor stack, or
re-derived locally) to all three output rasters before export.

**Note:** if a separate mangrove mask raster was not exported in
`01_preprocessing.ipynb`, valid (non-NaN) pixels in the predictor stack
already approximate the mangrove extent, since GEDI sampling was
restricted to masked pixels. Load an explicit mask raster here if available
for a cleaner edge.


In [ ]:
MASK_PATH = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/data/raw/gmw_mask_west_kalimantan.tif'
with rasterio.open(MASK_PATH) as src_mask:
    mask_array = src_mask.read(1)

ch_map     = ru.apply_mangrove_mask(ch_map,     mask_array)
agb_map    = ru.apply_mangrove_mask(agb_map,    mask_array)
carbon_map = ru.apply_mangrove_mask(carbon_map, mask_array)

print('GMW mangrove mask applied to CH, AGB, and carbon stock maps.')


## 7. Distribution Statistics

Pixel area at 25 m resolution = 625 m2 = 0.0625 ha. Total carbon stock is
the per-pixel sum scaled by pixel area, reported in kt C.


In [ ]:
PIXEL_AREA_HA = 0.0625  # 25 m x 25 m

ch_stats = ru.raster_summary_stats(ch_map, label='Canopy Height (m)')
print()
agb_stats = ru.raster_summary_stats(agb_map, label='AGB (Mg/ha)')
print()
carbon_stats = ru.raster_summary_stats(carbon_map, pixel_area_ha=PIXEL_AREA_HA, label='Carbon Stock (Mg C/ha)')

total_carbon_kt = carbon_stats['total'] / 1000
print(f'\nTotal carbon stock: {total_carbon_kt:.2f} kt C')


## 8. Visualization


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im0 = axes[0].imshow(ch_map, cmap='YlOrRd', vmin=0, vmax=25)
axes[0].set_title('Canopy Height (m)')
axes[0].axis('off')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(agb_map, cmap='Greens', vmin=0, vmax=200)
axes[1].set_title('AGB (Mg/ha)')
axes[1].axis('off')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(carbon_map, cmap='PuRd', vmin=0, vmax=90)
axes[2].set_title('Carbon Stock (Mg C/ha)')
axes[2].axis('off')
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig('/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/images/final_maps_ch_agb_carbon.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Export Final Rasters


In [ ]:
ru.write_geotiff(ch_map,     profile, '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/outputs/canopy_height_map_west_kalimantan.tif')
ru.write_geotiff(agb_map,    profile, '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/outputs/agb_map_west_kalimantan.tif')
ru.write_geotiff(carbon_map, profile, '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/outputs/carbon_stock_map_west_kalimantan.tif')

print('All final rasters exported to outputs/.')


## Summary

| Output | Path |
|--------|------|
| Canopy height map | `outputs/canopy_height_map_west_kalimantan.tif` |
| AGB map | `outputs/agb_map_west_kalimantan.tif` |
| Carbon stock map | `outputs/carbon_stock_map_west_kalimantan.tif` |

| Carbon Stock Stat | Value |
|--------------------|------:|
| Mean | see output above |
| Median | see output above |
| Std Dev | see output above |
| Min / Max | see output above |
| Total | see output above (kt C) |

This completes the Python/scikit-learn migration of the two-stage CH + AGB
+ carbon stock pipeline. Results can now be compared head-to-head against
the GEE `smileRandomForest` baseline documented in the parent GEE repo
README.
